In [1]:
# -------------------------------------------------------------------------
# 0. Dependencies Installation (IU X-Ray OOD Evaluation + Semantic Metrics)
# -------------------------------------------------------------------------
!pip install -q -U bitsandbytes transformers peft accelerate evaluate rouge-score nltk bert-score sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.1 MB/s eta 0:00:00


In [2]:
# -------------------------------------------------------------------------
# 1. Model 7 Out-of-Distribution (OOD) Evaluation on IU X-Ray Dataset
#    Zero-Shot Cross-Domain Transfer: Trained on MIMIC-CXR (Boston)
#    Evaluated on IU X-Ray (Indiana) without any fine-tuning
#    Beam Search (num_beams=4) | 2D Spatial Layout (16x8)
#    Perceiver Resampler (32 Queries) + Soft Classifier Conditioning
#    Evaluated with: NLP Metrics + Clinical Keywords + Semantic BERTScore (PubMedBERT)
# -------------------------------------------------------------------------
import os, sys, gc, glob, time, json, math, random, re, torch
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from PIL import Image
from torchvision import transforms

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
from nltk.util import ngrams
from rouge_score import rouge_scorer
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, set_peft_model_state_dict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"OOD Evaluation Compute Device: {device}", flush=True)

# =========================================================================
# PHASE 1: LOCATE MODEL 7 CHECKPOINT & THRESHOLDS
# =========================================================================
print("=" * 70, flush=True)
print("PHASE 1: Locating Model 7 Checkpoint & Optimal Thresholds", flush=True)
print("=" * 70, flush=True)

print("Scanning all mounted files in /kaggle/input...", flush=True)
all_mounted = []
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        full_path = os.path.join(root, f)
        if f.endswith(('.pt', '.pth', '.json', '.csv')):
            all_mounted.append(full_path)

if all_mounted:
    print(f"Found {len(all_mounted)} relevant files in /kaggle/input. Sample files:")
    for p in all_mounted[:8]:
        print(f"  -> {p}", flush=True)
else:
    print("WARNING: No .pt, .pth, .json, or .csv files found in /kaggle/input!", flush=True)

best_ckpts = []
latest_ckpts = []
other_ckpts = []

for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        f_lower = f.lower()
        full = os.path.join(root, f)
        if "model7_qwen_best.pt" in f_lower:
            best_ckpts.append(full)
        elif "model7_qwen_latest.pt" in f_lower:
            latest_ckpts.append(full)
        elif f_lower.endswith(".pt") and "model7" in f_lower:
            other_ckpts.append(full)
        elif f_lower.endswith(".pt") or f_lower.endswith(".pth"):
            other_ckpts.append(full)

checkpoint_path = None
if best_ckpts:
    best_meta = {"epoch": -1, "val_loss": float("inf")}
    for cand in best_ckpts:
        try:
            ckpt_dict = torch.load(cand, map_location="cpu")
            vl = ckpt_dict.get("val_loss", float("inf"))
            ep = ckpt_dict.get("epoch", 1)
            if vl < best_meta["val_loss"]:
                best_meta = {"epoch": ep, "val_loss": vl}
                checkpoint_path = cand
        except Exception:
            if checkpoint_path is None:
                checkpoint_path = cand
elif latest_ckpts:
    checkpoint_path = latest_ckpts[0]
elif other_ckpts:
    print(f"Found other checkpoint candidate: {other_ckpts[0]}", flush=True)
    checkpoint_path = other_ckpts[0]

if not checkpoint_path:
    print("\n" + "!" * 70, flush=True)
    print("ERROR: Model 7 checkpoint (.pt file) not found in /kaggle/input!", flush=True)
    if os.path.exists("/kaggle/input"):
        for d in os.listdir("/kaggle/input"):
            subpath = os.path.join("/kaggle/input", d)
            print(f"  /kaggle/input/{d} (items: {len(os.listdir(subpath)) if os.path.isdir(subpath) else 'file'})", flush=True)
    print("!" * 70 + "\n", flush=True)
    raise FileNotFoundError(
        "Model 7 checkpoint not found! Please click '+ Add Input' -> select your "
        "'mimic-cxr-model-7-train2' notebook run and click Add."
    )

print(f"Successfully Selected Checkpoint: {checkpoint_path}", flush=True)

# Load optimal thresholds
thresholds_path = None
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "optimal_thresholds.json":
            thresholds_path = os.path.join(root, f)
            break

CHEXPERT_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Enlarged Cardiomediastinum",
    "Fracture", "Lung Lesion", "Lung Opacity", "No Finding", "Pleural Effusion",
    "Pleural Other", "Pneumonia", "Pneumothorax", "Support Devices"
]

optimal_thresholds = {col: 0.5 for col in CHEXPERT_COLS}
if thresholds_path and os.path.exists(thresholds_path):
    with open(thresholds_path) as f:
        loaded_th = json.load(f)
        optimal_thresholds.update(loaded_th)
    print(f"Loaded class-specific optimal decision thresholds from: {thresholds_path}", flush=True)
else:
    print("Using default 0.5 decision thresholds.", flush=True)

# =========================================================================
# PHASE 2: LOAD & PARSE IU X-RAY DATASET (FAIR IMPRESSION TARGETING)
# =========================================================================
print("\n" + "=" * 70, flush=True)
print("PHASE 2: Loading & Parsing IU X-Ray Dataset (Out-of-Distribution)", flush=True)
print("=" * 70, flush=True)

iu_base_candidates = [
    "/kaggle/input/chest-xrays-indiana-university",
    "/kaggle/input/datasets/raddar/chest-xrays-indiana-university",
    "/kaggle/input/iu-chest-x-rays-cleaned",
]

iu_base_dir = None
for cand in iu_base_candidates:
    if os.path.exists(cand):
        iu_base_dir = cand
        break

if iu_base_dir is None:
    for root, _, files in os.walk("/kaggle/input"):
        if "indiana_reports.csv" in files:
            iu_base_dir = root
            break

if iu_base_dir is None:
    raise FileNotFoundError(
        "IU X-Ray dataset not found! Please add 'raddar/chest-xrays-indiana-university' "
        "from Kaggle Datasets via '+ Add Input'."
    )

print(f"IU X-Ray base directory: {iu_base_dir}", flush=True)

reports_csv = os.path.join(iu_base_dir, "indiana_reports.csv")
if not os.path.exists(reports_csv):
    for root, _, files in os.walk(iu_base_dir):
        if "indiana_reports.csv" in files:
            reports_csv = os.path.join(root, "indiana_reports.csv")
            break

iu_reports = pd.read_csv(reports_csv)
print(f"Reports CSV: {reports_csv}", flush=True)
print(f"Reports shape: {iu_reports.shape}, columns: {list(iu_reports.columns)}", flush=True)

proj_csv = os.path.join(iu_base_dir, "indiana_projections.csv")
if not os.path.exists(proj_csv):
    for root, _, files in os.walk(iu_base_dir):
        if "indiana_projections.csv" in files:
            proj_csv = os.path.join(root, "indiana_projections.csv")
            break

iu_projections = pd.read_csv(proj_csv)
print(f"Projections CSV: {proj_csv}", flush=True)

iu_images_dir = None
for candidate_subdir in ["images/images_normalized", "images", "images_normalized"]:
    test_path = os.path.join(iu_base_dir, candidate_subdir)
    if os.path.isdir(test_path):
        some_files = os.listdir(test_path)[:5]
        if any(f.lower().endswith((".png", ".jpg")) for f in some_files):
            iu_images_dir = test_path
            break

if iu_images_dir is None:
    for root, dirs, files in os.walk(iu_base_dir):
        png_count = sum(1 for f in files if f.lower().endswith((".png", ".jpg")))
        if png_count > 100:
            iu_images_dir = root
            break

print(f"Images directory: {iu_images_dir}", flush=True)

# 1. Filter Frontal views only and deduplicate by patient uid (one frontal study per patient)
frontal_df = iu_projections[iu_projections["projection"].str.lower().str.contains("frontal", na=False)].drop_duplicates(subset="uid").copy()
print(f"Unique frontal patient studies: {len(frontal_df)}", flush=True)

# 2. Merge reports with frontal projections
merged_df = pd.merge(frontal_df, iu_reports, on="uid", how="inner")
print(f"Merged (frontal + reports): {len(merged_df)}", flush=True)

# 3. Clean Text and Define Task-Aligned Targets:
# Model 7 was trained to generate the IMPRESSION (clinical conclusion).
# Therefore, comparing against Impression directly avoids the catastrophic length mismatch (Brevity Penalty).
def clean_iu_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    # Remove HIPAA de-identification redactions (e.g. XXXX-year-old, normal chest x-XXXX)
    text = re.sub(r'(?i)xxxx[- ]year[- ]old', '', text)
    text = re.sub(r'(?i)x-xxxx', 'x-ray', text)
    text = re.sub(r'(?i)\bxxxx\b', '', text)
    text = re.sub(r'X{2,}', '', text)
    text = re.sub(r'\b\d+\.\s*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def parse_ground_truth(row):
    findings_clean = clean_iu_text(str(row.get("findings", "")) if pd.notna(row.get("findings")) else "")
    impression_clean = clean_iu_text(str(row.get("impression", "")) if pd.notna(row.get("impression")) else "")
    
    # Primary Target: Impression (if present), else Findings
    primary_target = impression_clean if len(impression_clean) >= 5 else findings_clean
    # Full Target: Findings + Impression
    full_target = f"{findings_clean} {impression_clean}".strip() if (findings_clean and impression_clean) else (findings_clean or impression_clean)
    
    return primary_target, full_target

merged_df[["report", "full_report"]] = merged_df.apply(parse_ground_truth, axis=1, result_type="expand")

# Filter out empty reports
merged_df = merged_df[merged_df["report"].str.strip().str.len() >= 5].reset_index(drop=True)
print(f"Cohort size after text cleaning: {len(merged_df)}", flush=True)

# 4. Build and verify image paths
merged_df["image_path"] = merged_df["filename"].apply(lambda f: os.path.join(iu_images_dir, f))
valid_mask = merged_df["image_path"].apply(os.path.exists)
missing_count = (~valid_mask).sum()
if missing_count > 0:
    print(f"Filtering {missing_count} missing images...", flush=True)
    merged_df = merged_df[valid_mask].reset_index(drop=True)

iu_df = merged_df[["uid", "filename", "image_path", "report", "full_report"]].copy()
num_iu = len(iu_df)

print(f"\n{'='*50}", flush=True)
print(f"FINAL IU X-RAY OOD COHORT: {num_iu} verified frontal image-report pairs", flush=True)
print(f"{'='*50}", flush=True)

print("\nSample Ground Truth Reports (first 3):")
for idx in range(min(3, num_iu)):
    print(f"  [{idx}] UID={iu_df.iloc[idx]['uid']}: {iu_df.iloc[idx]['report']}")

# =========================================================================
# PHASE 3: RAD-DINO 2D SPATIAL FEATURE EXTRACTION (ON-THE-FLY)
# =========================================================================
print("\n" + "=" * 70, flush=True)
print("PHASE 3: Extracting RAD-DINO 2D Spatial Features for IU X-Ray", flush=True)
print("=" * 70, flush=True)

GRID_H, GRID_W = 16, 8

class RadDino2DExtractor(nn.Module):
    def __init__(self, grid_h=16, grid_w=8):
        super().__init__()
        self.model = AutoModel.from_pretrained("microsoft/rad-dino", trust_remote_code=True)
        self.model.eval()
        self.grid_h = grid_h
        self.grid_w = grid_w
        
    @torch.no_grad()
    def forward(self, x):
        with torch.amp.autocast("cuda"):
            outputs = self.model(x)
            hidden = outputs.last_hidden_state
            patch_tokens = hidden[:, 1:, :]
            
            B = patch_tokens.size(0)
            patch_2d = patch_tokens.view(B, 37, 37, 768).permute(0, 3, 1, 2)
            spatial_2d = F.adaptive_avg_pool2d(patch_2d, (self.grid_h, self.grid_w))
            spatial_128 = spatial_2d.permute(0, 2, 3, 1).reshape(B, self.grid_h * self.grid_w, 768)
            
        return spatial_128

raddino_extractor = RadDino2DExtractor(grid_h=GRID_H, grid_w=GRID_W).to(device)

rad_transform = transforms.Compose([
    transforms.Resize(518, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(518),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5307, 0.5307, 0.5307], std=[0.2583, 0.2583, 0.2583])
])

from torch.utils.data import Dataset, DataLoader

class IUFeatureDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.iloc[idx]["image_path"]
        try:
            img = Image.open(path).convert("RGB")
            tensor = self.transform(img)
        except Exception:
            tensor = torch.zeros(3, 518, 518)
        return tensor

iu_dataset = IUFeatureDataset(iu_df, rad_transform)
iu_loader = DataLoader(iu_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f"Extracting 2D-pooled RAD-DINO features for {num_iu} IU X-Ray images...", flush=True)
iu_features = np.zeros((num_iu, 128, 768), dtype=np.float16)
idx_cursor = 0

for batch_tensors in tqdm(iu_loader, desc="RAD-DINO IU Features"):
    b_size = batch_tensors.size(0)
    batch_tensors = batch_tensors.to(device)
    spatial_tokens = raddino_extractor(batch_tensors)
    iu_features[idx_cursor:idx_cursor + b_size] = spatial_tokens.cpu().to(torch.float16).numpy()
    idx_cursor += b_size

del raddino_extractor
torch.cuda.empty_cache()
gc.collect()
print(f"Feature extraction complete: {iu_features.shape}", flush=True)

# =========================================================================
# PHASE 4: LOAD MODEL 7 ARCHITECTURE & WEIGHTS
# =========================================================================
print("\n" + "=" * 70, flush=True)
print("PHASE 4: Loading Model 7 Architecture & Weights", flush=True)
print("=" * 70, flush=True)

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
    device_map={"": 0},
    trust_remote_code=True
)
base_qwen = prepare_model_for_kbit_training(base_qwen)

ckpt = torch.load(checkpoint_path, map_location="cpu")
ckpt_epoch = ckpt.get("epoch", "unknown")
ckpt_loss = ckpt.get("val_loss", "unknown")
print(f"Checkpoint Metadata: Epoch={ckpt_epoch} | Val Loss={ckpt_loss}", flush=True)

detected_rank = 32
for k, v in ckpt["lora_state"].items():
    if "lora_A" in k:
        detected_rank = v.shape[0]
        break

detected_alpha = detected_rank * 2
print(f"LoRA Config: r={detected_rank}, alpha={detected_alpha}", flush=True)

peft_config = LoraConfig(
    r=detected_rank,
    lora_alpha=detected_alpha,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
qwen_lora_model = get_peft_model(base_qwen, peft_config)

class AttentionPoolClassifier(nn.Module):
    def __init__(self, visual_dim=768, num_classes=14):
        super().__init__()
        self.num_classes = num_classes
        self.visual_dim = visual_dim
        
        self.attn_proj = nn.Sequential(
            nn.Linear(visual_dim, 256),
            nn.Tanh(),
            nn.Linear(256, num_classes)
        )
        
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(visual_dim, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(256, 1)
            ) for _ in range(num_classes)
        ])
        
    def forward(self, spatial_tokens):
        attn_scores = self.attn_proj(spatial_tokens)
        attn_weights = F.softmax(attn_scores, dim=1)
        pooled = torch.einsum('bpd,bpc->bcd', spatial_tokens, attn_weights)
        logits_list = [self.heads[c](pooled[:, c, :]) for c in range(self.num_classes)]
        logits = torch.cat(logits_list, dim=-1)
        return logits, attn_weights

class PerceiverResampler(nn.Module):
    def __init__(self, visual_dim=768, qwen_dim=2048, num_queries=32, num_heads=8, num_layers=2, dropout=0.1):
        super().__init__()
        self.num_queries = num_queries
        self.qwen_dim = qwen_dim
        
        self.input_proj = nn.Linear(visual_dim, qwen_dim)
        self.input_norm = nn.LayerNorm(qwen_dim)
        self.queries = nn.Parameter(torch.randn(1, num_queries, qwen_dim) * 0.02)
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=qwen_dim,
            nhead=num_heads,
            dim_feedforward=qwen_dim * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_norm = nn.LayerNorm(qwen_dim)
        
    def forward(self, visual_tokens):
        B = visual_tokens.size(0)
        memory = self.input_norm(self.input_proj(visual_tokens))
        queries = self.queries.expand(B, -1, -1)
        resampled = self.decoder(tgt=queries, memory=memory)
        return self.output_norm(resampled)

class MultiTaskModel7(nn.Module):
    def __init__(self, qwen_model, visual_dim=768, qwen_dim=2048, num_queries=32):
        super().__init__()
        self.qwen = qwen_model
        self.num_queries = num_queries
        self.num_prefix_tokens = num_queries + 1
        self.qwen_dim = qwen_dim
        
        self.perceiver_resampler = PerceiverResampler(
            visual_dim=visual_dim, qwen_dim=qwen_dim, num_queries=num_queries, num_layers=2
        ).to(device=device)
        
        self.attn_pool_classifier = AttentionPoolClassifier(
            visual_dim=visual_dim, num_classes=14
        ).to(device=device)
        
        self.cls_embed_proj = nn.Sequential(
            nn.Linear(14, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, qwen_dim),
            nn.LayerNorm(qwen_dim)
        ).to(device=device)
        
    @torch.inference_mode()
    def generate_report(self, spatial_token, max_new_tokens=150, num_beams=4):
        self.eval()
        self.qwen.config.use_cache = True
        
        if spatial_token.dim() == 2:
            spatial_token = spatial_token.unsqueeze(0)
            
        # Step A: Disease Classifier Prediction with Attention Pooling
        cls_logits, attn_weights = self.attn_pool_classifier(spatial_token.float())
        cls_probs_t = torch.sigmoid(cls_logits.float())
        cls_probs = cls_probs_t.squeeze(0).cpu().numpy()
        
        active_findings = []
        for c_idx, c_name in enumerate(CHEXPERT_COLS):
            thresh = optimal_thresholds.get(c_name, 0.5)
            if cls_probs[c_idx] >= thresh:
                active_findings.append(c_name)
                
        if not active_findings:
            pathology_str = "No acute cardiopulmonary abnormality"
        else:
            pathology_str = ", ".join(active_findings)
            
        # Step B: Build Zero-Shot Prompt (Reference: None.)
        prompt_str = f"Pathology Assessment: {pathology_str}\nReference: None.\nImpression:"
        prompt_enc = tokenizer(prompt_str, return_tensors="pt").to(device)
        prompt_ids = prompt_enc["input_ids"]
        prompt_embeds = self.qwen.get_input_embeddings()(prompt_ids)
        
        # Step C: Multimodal Prefix (32 Perceiver Queries + 1 Soft Disease Token)
        visual_queries = self.perceiver_resampler(spatial_token.float())
        cls_token = self.cls_embed_proj(cls_probs_t).unsqueeze(1)
        
        if visual_queries.dtype != prompt_embeds.dtype:
            visual_queries = visual_queries.to(prompt_embeds.dtype)
        if cls_token.dtype != prompt_embeds.dtype:
            cls_token = cls_token.to(prompt_embeds.dtype)
            
        prefix_embeds = torch.cat([visual_queries, cls_token], dim=1)
        inputs_embeds = torch.cat([prefix_embeds, prompt_embeds], dim=1)
        
        # Step D: Clinically Safe Beam Search Decoding
        attn_mask_2d = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=device)
        
        gen_out = self.qwen.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attn_mask_2d,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True,
            repetition_penalty=1.05,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True
        )
        
        if gen_out.size(1) > inputs_embeds.size(1):
            gen_tokens = gen_out[0, inputs_embeds.size(1):]
        else:
            gen_tokens = gen_out[0]
        pred_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        
        return pred_text, cls_probs, attn_weights.squeeze(0).cpu().numpy(), pathology_str

model = MultiTaskModel7(qwen_lora_model).to(device)
model.perceiver_resampler.load_state_dict(ckpt["perceiver_resampler"])
model.cls_embed_proj.load_state_dict(ckpt["cls_embed_proj"])
model.attn_pool_classifier.load_state_dict(ckpt["attn_pool_classifier"])
set_peft_model_state_dict(qwen_lora_model, ckpt["lora_state"])
model.eval()
print(f"Model 7 loaded successfully. Ready for OOD evaluation.", flush=True)

del ckpt
torch.cuda.empty_cache()
gc.collect()

# =========================================================================
# PHASE 5: OOD EVALUATION & GENERATION LOOP
# =========================================================================
print("\n" + "=" * 70, flush=True)
print(f"PHASE 5: Zero-Shot OOD Evaluation on {num_iu} IU X-Ray Samples", flush=True)
print("=" * 70, flush=True)

# NegEx Clinical Concept Extractor
clinical_keywords = {
    'Atelectasis': ['atelectasis', 'atelectatic', 'collapse', 'volume loss'],
    'Cardiomegaly': ['cardiomegaly', 'enlarged heart', 'enlarged cardiac', 'cardiac enlargement'],
    'Consolidation': ['consolidation', 'airspace disease', 'airspace opacity'],
    'Edema': ['edema', 'vascular congestion', 'fluid overload', 'interstitial edema'],
    'Enlarged Cardiomediastinum': ['widened mediastinum', 'mediastinal widening', 'cardiomediastinum'],
    'Fracture': ['fracture', 'fractured', 'broken rib', 'healing fracture'],
    'Lung Lesion': ['lung lesion', 'nodule', 'mass', 'nodular opacity'],
    'Lung Opacity': ['opacity', 'opacities', 'infiltrate', 'clouding'],
    'No Finding': ['normal', 'clear', 'unremarkable', 'no acute', 'negative'],
    'Pleural Effusion': ['pleural effusion', 'effusion', 'blunting', 'fluid in pleural'],
    'Pleural Other': ['pleural thickening', 'fibrosis', 'calcification'],
    'Pneumonia': ['pneumonia', 'infectious process', 'bronchopneumonia'],
    'Pneumothorax': ['pneumothorax', 'air in pleural', 'apical pneumothorax'],
    'Support Devices': ['tube', 'catheter', 'pacemaker', 'pacer', 'line', 'picc', 'stent']
}

negation_prefix_pattern = re.compile(r"\b(no|without|free of|negative for|absence of|denies|cleared of)\b\s*([\w\s]{0,35})$", re.IGNORECASE)
negation_suffix_pattern = re.compile(r"^([\w\s]{0,35})\s*\b(ruled out|is absent|resolved|excluded|unlikely|is not seen|not identified|has resolved|is clear)\b", re.IGNORECASE)

def detect_clinical_presence(text, keywords):
    text_lower = text.lower()
    for kw in keywords:
        matches = [m.start() for m in re.finditer(re.escape(kw), text_lower)]
        for match_idx in matches:
            prefix_window = text_lower[max(0, match_idx - 35):match_idx]
            suffix_window = text_lower[match_idx + len(kw):min(len(text_lower), match_idx + len(kw) + 35)]
            if not negation_prefix_pattern.search(prefix_window) and not negation_suffix_pattern.search(suffix_window):
                return 1
    return 0

smooth = SmoothingFunction().method1
rouge_evaluator = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

results = []
bleu1_s, bleu2_s, bleu3_s, bleu4_s = [], [], [], []
rouge1_s, rouge2_s, rougeL_s = [], [], []
distinct1_s, distinct2_s = [], []
gt_pathology_list = []
pred_pathology_list = []
corpus_refs = []
corpus_hyps = []
xai_samples = []

start_eval_time = time.time()
print(f"Starting Zero-Shot OOD evaluation (Beam Search k=4, Target=Impression)...", flush=True)

for i in range(num_iu):
    row = iu_df.iloc[i]
    spatial_t = torch.from_numpy(np.array(iu_features[i])).to(device)
    gt_text = str(row["report"]).strip()
    
    pred_text, cls_probs, attn_maps, assessed_str = model.generate_report(
        spatial_t, max_new_tokens=150, num_beams=4
    )
    
    # Save candidates for XAI visualization
    if len(xai_samples) < 8:
        xai_samples.append({
            "idx": i,
            "uid": str(row["uid"]),
            "gt_text": gt_text,
            "pred_text": pred_text,
            "assessed_pathologies": assessed_str,
            "cls_probs": cls_probs,
            "attn_maps": attn_maps
        })
    
    ref_tok = [nltk.word_tokenize(gt_text.lower())] if gt_text else [[""]]
    hyp_tok = nltk.word_tokenize(pred_text.lower()) if pred_text else [""]
    
    corpus_refs.append(ref_tok)
    corpus_hyps.append(hyp_tok)
    
    b1 = sentence_bleu(ref_tok, hyp_tok, weights=(1, 0, 0, 0), smoothing_function=smooth)
    b2 = sentence_bleu(ref_tok, hyp_tok, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
    b3 = sentence_bleu(ref_tok, hyp_tok, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth)
    b4 = sentence_bleu(ref_tok, hyp_tok, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
    
    rouge_res = rouge_evaluator.score(gt_text, pred_text)
    r1 = rouge_res['rouge1'].fmeasure
    r2 = rouge_res['rouge2'].fmeasure
    rL = rouge_res['rougeL'].fmeasure
    
    bleu1_s.append(b1)
    bleu2_s.append(b2)
    bleu3_s.append(b3)
    bleu4_s.append(b4)
    rouge1_s.append(r1)
    rouge2_s.append(r2)
    rougeL_s.append(rL)
    
    if len(hyp_tok) >= 1:
        distinct1_s.append(len(set(hyp_tok)) / len(hyp_tok))
    else:
        distinct1_s.append(1.0)
        
    if len(hyp_tok) >= 2:
        bigrams = list(ngrams(hyp_tok, 2))
        distinct2_s.append(len(set(bigrams)) / len(bigrams))
    else:
        distinct2_s.append(1.0)
        
    gt_labels = [detect_clinical_presence(gt_text, kws) for kws in clinical_keywords.values()]
    pred_labels = [detect_clinical_presence(pred_text, kws) for kws in clinical_keywords.values()]
    
    gt_pathology_list.append(gt_labels)
    pred_pathology_list.append(pred_labels)
    
    results.append({
        "uid": str(row["uid"]),
        "ground_truth": gt_text,
        "prediction": pred_text,
        "bleu1": round(b1, 4),
        "bleu4": round(b4, 4),
        "rougeL": round(rL, 4)
    })
    
    # Live preview in the first 5 samples so you see real outputs immediately!
    if i < 5:
        print(f"\n--- Live Preview [Sample {i+1}/{num_iu}] ---", flush=True)
        print(f"  Disease Classifier : {assessed_str}", flush=True)
        print(f"  Ground Truth (IU)  : {gt_text}", flush=True)
        print(f"  Model Prediction   : {pred_text}", flush=True)
        print(f"  Sample BLEU-1: {b1:.4f} | BLEU-4: {b4:.4f} | ROUGE-L: {rL:.4f}", flush=True)
    
    if (i + 1) % 100 == 0 or (i + 1) == num_iu:
        elapsed = (time.time() - start_eval_time) / 60.0
        pct = ((i + 1) / num_iu) * 100.0
        eta = (elapsed / (i + 1)) * (num_iu - (i + 1))
        avg_b1 = np.mean(bleu1_s)
        avg_b4 = np.mean(bleu4_s)
        avg_rl = np.mean(rougeL_s)
        print(f"[OOD Eval {i+1:04d}/{num_iu} ({pct:5.1f}%)] Elapsed: {elapsed:.1f}m | ETA: {eta:.1f}m | BLEU-1: {avg_b1:.4f} | BLEU-4: {avg_b4:.4f} | ROUGE-L: {avg_rl:.4f}", flush=True)
        
    if (i + 1) % 250 == 0:
        torch.cuda.empty_cache()

# Free Qwen model memory before semantic evaluation
del model, qwen_lora_model, base_qwen
torch.cuda.empty_cache()
gc.collect()
print("\nGeneration complete. Freed LLM memory for semantic metrics.", flush=True)

# =========================================================================
# PHASE 6: SEMANTIC EVALUATION (Clinical BERTScore & SBERT Cosine Similarity)
# =========================================================================
print("\n" + "=" * 70, flush=True)
print("PHASE 6: Computing Modern Clinical Semantic Metrics", flush=True)
print("=" * 70, flush=True)

all_preds = [r["prediction"] for r in results]
all_refs = [r["ground_truth"] for r in results]

# 1. Clinical BERTScore (using Microsoft PubMedBERT)
print("1. Computing Clinical BERTScore with Microsoft PubMedBERT...", flush=True)
try:
    from bert_score import score as bert_score_eval
    P_b, R_b, F1_b = bert_score_eval(
        all_preds, all_refs,
        model_type="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
        device=device,
        batch_size=64,
        verbose=False
    )
    clinical_bert_f1 = float(F1_b.mean().item())
    clinical_bert_precision = float(P_b.mean().item())
    clinical_bert_recall = float(R_b.mean().item())
    print(f"  -> Clinical BERTScore F1 (PubMedBERT) : {clinical_bert_f1:.4f}", flush=True)
    print(f"  -> Clinical BERTScore Rec (PubMedBERT): {clinical_bert_recall:.4f}", flush=True)
except Exception as e:
    print(f"  Warning: BERTScore computation encountered: {e}. Using fallback.", flush=True)
    clinical_bert_f1 = 0.0
    clinical_bert_precision = 0.0
    clinical_bert_recall = 0.0

# 2. Semantic Embedding Cosine Similarity (Sentence-Transformers)
print("2. Computing Sentence Embedding Semantic Cosine Similarity...", flush=True)
try:
    from sentence_transformers import SentenceTransformer, util
    sbert = SentenceTransformer('all-MiniLM-L6-v2', device=str(device))
    emb_preds = sbert.encode(all_preds, convert_to_tensor=True, batch_size=128)
    emb_refs = sbert.encode(all_refs, convert_to_tensor=True, batch_size=128)
    cos_sims = util.cos_sim(emb_preds, emb_refs).diagonal().cpu().numpy()
    semantic_cosine_sim = float(np.mean(cos_sims))
    print(f"  -> Semantic Cosine Similarity (SBERT) : {semantic_cosine_sim:.4f}", flush=True)
    del sbert, emb_preds, emb_refs
    torch.cuda.empty_cache()
except Exception as e:
    print(f"  Warning: SBERT computation encountered: {e}. Using fallback.", flush=True)
    semantic_cosine_sim = 0.0

# 3. Clinical Pathology Diagnoses (NegEx)
y_true = np.array(gt_pathology_list)
y_pred = np.array(pred_pathology_list)

pathology_names = list(clinical_keywords.keys())
core10_indices = [i for i, name in enumerate(pathology_names) if name not in [
    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Pleural Other'
]]

macro_f1_all14 = f1_score(y_true, y_pred, average="macro", zero_division=0)
macro_f1_core10 = f1_score(y_true[:, core10_indices], y_pred[:, core10_indices], average="macro", zero_division=0)

micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
micro_precision = precision_score(y_true, y_pred, average="micro", zero_division=0)
micro_recall = recall_score(y_true, y_pred, average="micro", zero_division=0)

corpus_b1 = corpus_bleu(corpus_refs, corpus_hyps, weights=(1, 0, 0, 0))
corpus_b4 = corpus_bleu(corpus_refs, corpus_hyps, weights=(0.25, 0.25, 0.25, 0.25))

clinical_diagnostic_table = []
for idx, name in enumerate(pathology_names):
    yt = y_true[:, idx]
    yp = y_pred[:, idx]
    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(yt, yp, average="binary", zero_division=0)
    
    clinical_diagnostic_table.append({
        "Pathology": name,
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Sensitivity": round(sensitivity, 4),
        "Specificity": round(specificity, 4),
        "F1": round(f1, 4)
    })

ood_metrics = {
    "Dataset": "IU X-Ray (OOD Zero-Shot)",
    "N_Samples": int(num_iu),
    "Sentence_BLEU-1": float(np.mean(bleu1_s)),
    "Sentence_BLEU-2": float(np.mean(bleu2_s)),
    "Sentence_BLEU-3": float(np.mean(bleu3_s)),
    "Sentence_BLEU-4": float(np.mean(bleu4_s)),
    "Corpus_BLEU-1": float(corpus_b1),
    "Corpus_BLEU-4": float(corpus_b4),
    "ROUGE-1": float(np.mean(rouge1_s)),
    "ROUGE-2": float(np.mean(rouge2_s)),
    "ROUGE-L": float(np.mean(rougeL_s)),
    "Clinical_BERTScore_F1": float(clinical_bert_f1),
    "Clinical_BERTScore_Recall": float(clinical_bert_recall),
    "Semantic_Cosine_Similarity": float(semantic_cosine_sim),
    "Distinct-1": float(np.mean(distinct1_s)),
    "Distinct-2": float(np.mean(distinct2_s)),
    "Clinical_Macro_F1_All14": float(macro_f1_all14),
    "Clinical_Macro_F1_Core10": float(macro_f1_core10),
    "Clinical_Micro_F1": float(micro_f1),
    "Clinical_Micro_Recall": float(micro_recall),
    "Evaluation_Duration_Minutes": float((time.time() - start_eval_time) / 60.0)
}

# MIMIC-CXR Reference Metrics
iid_metrics = {
    "Sentence_BLEU-1": 0.3046,
    "Sentence_BLEU-4": 0.1604,
    "ROUGE-1": 0.3382,
    "ROUGE-L": 0.3154,
    "Clinical_BERTScore_F1": 0.8820,
    "Semantic_Cosine_Similarity": 0.8250,
    "Distinct-2": 0.9952,
    "Clinical_Macro_F1_Core10": 0.4505,
    "Clinical_Micro_F1": 0.5210,
    "Clinical_Micro_Recall": 0.5547
}

print("\n" + "=" * 75, flush=True)
print(f"FINAL 360-DEGREE OOD BENCHMARK: IU X-Ray (N = {num_iu:,})", flush=True)
print("=" * 75, flush=True)

compare_keys = [
    "Sentence_BLEU-1", "Sentence_BLEU-4", "ROUGE-1", "ROUGE-L",
    "Clinical_BERTScore_F1", "Semantic_Cosine_Similarity",
    "Distinct-2", "Clinical_Macro_F1_Core10", "Clinical_Micro_F1", "Clinical_Micro_Recall"
]

print(f"  {'Metric':<28} | {'MIMIC (IID)':>11} | {'IU (OOD)':>11} | {'Evaluation Domain':>18}")
print("  " + "-" * 75)
for k in compare_keys:
    iid_v = iid_metrics.get(k, 0.0)
    ood_v = ood_metrics.get(k, 0.0)
    if "BERTScore" in k or "Semantic" in k:
        domain_tag = "Semantic (Medical)"
    elif "Clinical" in k:
        domain_tag = "Clinical Accuracy"
    else:
        domain_tag = "Lexical (N-gram)"
    print(f"  {k:<28} | {iid_v:>11.4f} | {ood_v:>11.4f} | {domain_tag:>18}")

# =========================================================================
# PHASE 7: PUBLICATION-GRADE VISUALIZATIONS
# =========================================================================
print("\n" + "=" * 70, flush=True)
print("PHASE 7: Generating Publication Visualizations", flush=True)
print("=" * 70, flush=True)

# 1. Radar Comparison Chart (Lexical + Semantic + Clinical)
radar_labels = ["BLEU-4", "ROUGE-L", "BERTScore\n(PubMed)", "Semantic\nSimilarity", "Clinical\nF1 (Core10)", "Clinical\nRecall"]
r_iid = [iid_metrics["Sentence_BLEU-4"], iid_metrics["ROUGE-L"], iid_metrics["Clinical_BERTScore_F1"],
         iid_metrics["Semantic_Cosine_Similarity"], iid_metrics["Clinical_Macro_F1_Core10"], iid_metrics["Clinical_Micro_Recall"]]
r_ood = [ood_metrics["Sentence_BLEU-4"], ood_metrics["ROUGE-L"], ood_metrics["Clinical_BERTScore_F1"],
         ood_metrics["Semantic_Cosine_Similarity"], ood_metrics["Clinical_Macro_F1_Core10"], ood_metrics["Clinical_Micro_Recall"]]

N_dim = len(radar_labels)
angles = [n / float(N_dim) * 2 * np.pi for n in range(N_dim)]
angles += angles[:1]
r_iid += r_iid[:1]
r_ood += r_ood[:1]

fig, ax = plt.subplots(figsize=(8.5, 8.5), subplot_kw=dict(polar=True))
ax.fill(angles, r_iid, alpha=0.18, color='royalblue')
ax.plot(angles, r_iid, 'o-', linewidth=2.5, color='royalblue', label='MIMIC-CXR (In-Distribution, N=6,388)')
ax.fill(angles, r_ood, alpha=0.18, color='crimson')
ax.plot(angles, r_ood, 's--', linewidth=2.5, color='crimson', label=f'IU X-Ray (Zero-Shot OOD, N={num_iu:,})')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend(loc='upper right', bbox_to_anchor=(1.38, 1.12), fontsize=11)
ax.set_title("Model 7 Cross-Domain Robustness Analysis\n(Lexical, Semantic & Clinical)", fontsize=13, fontweight='bold', pad=25)
plt.tight_layout()
plt.savefig("/kaggle/working/ood_radar_comparison.png", dpi=300, bbox_inches="tight")
plt.close()

# 2. Side-by-side Bar Chart
fig, ax = plt.subplots(figsize=(13, 5.5))
bar_keys = ["BLEU-4", "ROUGE-L", "BERTScore F1", "Semantic Sim", "Clinical F1 (Core10)", "Clinical Recall"]
b_iid = [iid_metrics["Sentence_BLEU-4"], iid_metrics["ROUGE-L"], iid_metrics["Clinical_BERTScore_F1"],
         iid_metrics["Semantic_Cosine_Similarity"], iid_metrics["Clinical_Macro_F1_Core10"], iid_metrics["Clinical_Micro_Recall"]]
b_ood = [ood_metrics["Sentence_BLEU-4"], ood_metrics["ROUGE-L"], ood_metrics["Clinical_BERTScore_F1"],
         ood_metrics["Semantic_Cosine_Similarity"], ood_metrics["Clinical_Macro_F1_Core10"], ood_metrics["Clinical_Micro_Recall"]]

x_pos = np.arange(len(bar_keys))
w = 0.35
bars1 = ax.bar(x_pos - w/2, b_iid, w, label='MIMIC-CXR (IID)', color='royalblue', alpha=0.85)
bars2 = ax.bar(x_pos + w/2, b_ood, w, label=f'IU X-Ray (OOD Zero-Shot)', color='crimson', alpha=0.85)

for b in bars1:
    ax.text(b.get_x() + b.get_width()/2., b.get_height() + 0.015, f'{b.get_height():.3f}',
            ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='royalblue')
for b in bars2:
    ax.text(b.get_x() + b.get_width()/2., b.get_height() + 0.015, f'{b.get_height():.3f}',
            ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='crimson')

ax.set_xticks(x_pos)
ax.set_xticklabels(bar_keys, fontsize=10.5, fontweight='bold')
ax.set_ylabel("Score (0.0 - 1.0)", fontsize=11, fontweight='bold')
ax.set_title(f"Model 7 In-Distribution vs Zero-Shot Out-of-Distribution Generalization", fontsize=13, fontweight='bold')
ax.legend(fontsize=10.5, loc='upper right')
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/ood_bar_comparison.png", dpi=300, bbox_inches="tight")
plt.close()

# 3. Confusion Matrix
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()
for idx, name in enumerate(pathology_names):
    cm = np.array([
        [clinical_diagnostic_table[idx]["TN"], clinical_diagnostic_table[idx]["FP"]],
        [clinical_diagnostic_table[idx]["FN"], clinical_diagnostic_table[idx]["TP"]]
    ])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", cbar=False, ax=axes[idx],
                xticklabels=["Pred Neg", "Pred Pos"], yticklabels=["Actual Neg", "Actual Pos"])
    axes[idx].set_title(f"{name}\n(Sens: {clinical_diagnostic_table[idx]['Sensitivity']:.2f} | F1: {clinical_diagnostic_table[idx]['F1']:.2f})", fontsize=10, fontweight="bold")

for j in range(len(pathology_names), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(f"Model 7 OOD: 14 Pathology Confusion Matrices on IU X-Ray (N={num_iu:,})", fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
plt.savefig("/kaggle/working/ood_pathology_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.close()

# 4. XAI Heatmaps
if xai_samples:
    focus_diseases = ["Cardiomegaly", "Pleural Effusion", "Pneumonia", "Atelectasis"]
    focus_indices = [pathology_names.index(d) for d in focus_diseases]
    n_show = min(4, len(xai_samples))
    fig, axes = plt.subplots(n_show, 4, figsize=(14, 4.5 * n_show))
    if n_show == 1:
        axes = np.expand_dims(axes, 0)
    
    for row_idx in range(n_show):
        sample = xai_samples[row_idx]
        for col_idx, d_idx in enumerate(focus_indices):
            grid_attn = sample["attn_maps"][:, d_idx].reshape(16, 8)
            grid_norm = (grid_attn - grid_attn.min()) / (grid_attn.max() - grid_attn.min() + 1e-8)
            prob_score = sample["cls_probs"][d_idx]
            sns.heatmap(grid_norm, cmap="magma", cbar=False, ax=axes[row_idx, col_idx])
            axes[row_idx, col_idx].set_title(f"{focus_diseases[col_idx]} (P={prob_score:.2f})", fontsize=10, fontweight="bold")
            axes[row_idx, col_idx].axis("off")
            
    plt.suptitle("Model 7 OOD: Explainable AI Attention Pooling Spatial Heatmaps (IU X-Ray)", fontsize=14, fontweight="bold", y=0.99)
    plt.tight_layout()
    plt.savefig("/kaggle/working/ood_attention_heatmaps.png", dpi=300, bbox_inches="tight")
    plt.close()

print("Publication figures saved successfully.", flush=True)

# =========================================================================
# PHASE 8: EXPORT ARTIFACTS & THESIS LATEX TABLE
# =========================================================================
print("\n" + "=" * 70, flush=True)
print("PHASE 8: Exporting All OOD Artifacts & LaTeX Table", flush=True)
print("=" * 70, flush=True)

def to_serializable(val):
    if hasattr(val, "item"):
        return val.item()
    if isinstance(val, (np.integer, np.int64)):
        return int(val)
    if isinstance(val, (np.floating, np.float32, np.float64)):
        return float(val)
    if isinstance(val, np.ndarray):
        return val.tolist()
    return str(val)

with open("/kaggle/working/model7_ood_metrics.json", "w") as f:
    json.dump(ood_metrics, f, indent=2, default=to_serializable)

pd.DataFrame(results).to_csv("/kaggle/working/model7_ood_predictions.csv", index=False)
pd.DataFrame(clinical_diagnostic_table).to_csv("/kaggle/working/model7_ood_clinical_diagnostics.csv", index=False)

# Thesis LaTeX Table
b1_str = f"{ood_metrics['Sentence_BLEU-1']:.4f}"
b4_str = f"{ood_metrics['Sentence_BLEU-4']:.4f}"
rl_str = f"{ood_metrics['ROUGE-L']:.4f}"
bert_str = f"{ood_metrics['Clinical_BERTScore_F1']:.4f}"
cos_str = f"{ood_metrics['Semantic_Cosine_Similarity']:.4f}"
f1c_str = f"{ood_metrics['Clinical_Macro_F1_Core10']:.4f}"
rcm_str = f"{ood_metrics['Clinical_Micro_Recall']:.4f}"

latex_lines = [
    r"\begin{table*}[htbp]",
    r"\centering",
    rf"\caption{{Cross-Domain Zero-Shot Generalization: Model 7 on Indiana University Chest X-Ray (N = {num_iu:,})}}",
    r"\label{tab:ood_generalization}",
    r"\resizebox{\textwidth}{!}{",
    r"\begin{tabular}{lccccccc}",
    r"\hline",
    r"\textbf{Evaluation Cohort} & \textbf{BLEU-1} & \textbf{BLEU-4} & \textbf{ROUGE-L} & \textbf{BERTScore (PubMed)} & \textbf{Semantic CosSim} & \textbf{Macro F1 (Core10)} & \textbf{Micro Recall} \\",
    r"\hline",
    r"MIMIC-CXR (In-Distribution, N=6,388) & 0.3046 & 0.1604 & 0.3154 & 0.8820 & 0.8250 & 0.4505 & 0.5547 \\",
    rf"\textbf{{IU X-Ray (Zero-Shot OOD, N={num_iu:,})}} & \textbf{{{b1_str}}} & \textbf{{{b4_str}}} & \textbf{{{rl_str}}} & \textbf{{{bert_str}}} & \textbf{{{cos_str}}} & \textbf{{{f1c_str}}} & \textbf{{{rcm_str}}} \\",
    r"\hline",
    r"\end{tabular}",
    r"}",
    r"\end{table*}"
]

with open("/kaggle/working/ood_comparison_table.tex", "w") as f:
    f.write("\n".join(latex_lines))

print("Automated Thesis LaTeX Table saved to /kaggle/working/ood_comparison_table.tex", flush=True)
print("\n" + "=" * 80, flush=True)
print("MODEL 7 OUT-OF-DISTRIBUTION BENCHMARK COMPLETED SUCCESSFULLY!")
print("All artifacts, figures, semantic metrics and tables saved in /kaggle/working")
print("=" * 80, flush=True)


OOD Evaluation Compute Device: cuda
PHASE 1: Locating Model 7 Checkpoint & Optimal Thresholds
Scanning all mounted files in /kaggle/input...
Found 8 relevant files in /kaggle/input. Sample files:
  -> /kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_projections.csv
  -> /kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_reports.csv
  -> /kaggle/input/notebooks/kimiasafaralian/mimic-cxr-model-7-train2/optimal_thresholds.json
  -> /kaggle/input/notebooks/kimiasafaralian/mimic-cxr-model-7-train2/model7_qwen_best.pt
  -> /kaggle/input/notebooks/kimiasafaralian/mimic-cxr-model-7-train2/__huggingface_repos__.json
  -> /kaggle/input/notebooks/kimiasafaralian/mimic-cxr-model-7-train2/training_history.json
  -> /kaggle/input/notebooks/kimiasafaralian/mimic-cxr-model-7-train2/model7_qwen_latest.pt
  -> /kaggle/input/notebooks/kimiasafaralian/mimic-cxr-model-7-train2/__output__.json
Successfully Selected Checkpoint: /kaggle/input/notebooks/kimiasafaralian/mim

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Extracting 2D-pooled RAD-DINO features for 3666 IU X-Ray images...


RAD-DINO IU Features:   0%|          | 0/230 [00:00<?, ?it/s]

... (23 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  -> Semantic Cosine Similarity (SBERT) : 0.5859

FINAL 360-DEGREE OOD BENCHMARK: IU X-Ray (N = 3,666)
  Metric                       | MIMIC (IID) |    IU (OOD) |  Evaluation Domain
  ---------------------------------------------------------------------------
  Sentence_BLEU-1              |      0.3046 |      0.3812 |   Lexical (N-gram)
  Sentence_BLEU-4              |      0.1604 |      0.1270 |   Lexical (N-gram)
  ROUGE-1                      |      0.3382 |      0.3889 |   Lexical (N-gram)
  ROUGE-L                      |      0.3154 |      0.3833 |   Lexical (N-gram)
  Clinical_BERTScore_F1        |      0.8820 |      0.0000 | Semantic (Medical)
  Semantic_Cosine_Similarity   |      0.8250 |      0.5859 | Semantic (Medical)
  Distinct-2                   |      0.9952 |      0.9992 |   Lexical (N-gram)
  Clinical_Macro_F1_Core10     |      0.4505 |      0.3920 |  Clinical Accuracy
  Clinical_Micro_F1            |      0.5210 |      0.6512 |  Clinical Accuracy
  Clinical_Micro_Re